# 06 — SLM Clinical Narrative

**Assignment 2: Explainable Maternal Health Risk Prediction** · KQC7016 Data Analytics

The final layer turns the structured evidence from notebook 05 (prediction + SHAP drivers +
matched rules) into a short, readable **clinical narrative** using a Small Language Model
(SLM). The SLM is a **constrained explanation layer — it does NOT predict**; it only
rewrites the evidence the classifier and rule miner already produced.

**Models compared (per plan §9):**
| Model | Size | Notes |
|---|---|---|
| `meta-llama/Llama-3.2-1B-Instruct` | 1B | lightweight baseline (gated — needs HF token) |
| `Qwen/Qwen3-1.7B` | 1.7B | efficiency/quality balance (ungated) |
| `meta-llama/Llama-3.2-3B-Instruct` | 3B | stronger candidate, 4-bit quantised (gated) |

> **Hardware:** RTX 3050 Laptop, 4 GB VRAM. Models are loaded one at a time and freed
> between runs. The 1.7B and 3B models use 4-bit (nf4) quantisation to fit the 4 GB budget;
> the 1B model runs in fp16.

> **Gated models:** the two Llama models require accepting their license on Hugging Face and
> a local token (`hf auth login`). Each model load is wrapped in `try/except`, so if the
> token is missing the notebook still runs (those models are marked *skipped*) — add the
> token and re-run to fill them in. Qwen3-1.7B is ungated and runs as-is.

In [1]:
import json, gc, time
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)

evidence = json.loads(Path("data/patient_evidence.json").read_text())
print(f"Loaded {len(evidence)} patient evidence records")
print("Risk classes present:", [e["predicted_risk"] for e in evidence])

MODELS = [
    {"name": "Llama-3.2-1B", "id": "meta-llama/Llama-3.2-1B-Instruct", "quant": False, "qwen": False},
    {"name": "Qwen3-1.7B",   "id": "Qwen/Qwen3-1.7B",                  "quant": True,  "qwen": True},
    {"name": "Llama-3.2-3B", "id": "meta-llama/Llama-3.2-3B-Instruct", "quant": True,  "qwen": False},
]

device: cuda | torch 2.6.0+cu118
Loaded 3 patient evidence records
Risk classes present: ['high', 'mid', 'low']


## 1. Prompt template

A fixed system prompt enforces the safety constraints and output format; the user prompt is
built deterministically from each patient's evidence. The five required sections are:
**Prediction · Main contributing factors · Rule-based support · Clinical interpretation ·
Safety note.**

In [2]:
SYSTEM_PROMPT = (
    "You are a clinical decision-support assistant for maternal health monitoring. "
    "You convert structured model evidence into a short, clear narrative for a clinician.\n"
    "STRICT RULES:\n"
    "1. Use ONLY the evidence provided. Do not invent values, history, or facts.\n"
    "2. Do NOT diagnose, prescribe, or recommend specific treatment.\n"
    "3. This is decision support only; the clinician makes all final decisions.\n"
    "4. Output EXACTLY these five sections, each on its own line with the heading followed "
    "by a colon: 'Prediction:', 'Main contributing factors:', 'Rule-based support:', "
    "'Clinical interpretation:', 'Safety note:'. Be concise (1-3 sentences per section)."
)

SECTIONS = ["Prediction", "Main contributing factors", "Rule-based support",
            "Clinical interpretation", "Safety note"]

UNITS = {"Age": "years", "SystolicBP": "mmHg", "DiastolicBP": "mmHg",
         "BS": "mmol/L", "BodyTemp": "F", "HeartRate": "bpm"}

def build_user_prompt(ev):
    v = ev["vitals"]
    lines = ["PATIENT EVIDENCE", ""]
    lines.append(f"Predicted risk level: {ev['predicted_risk'].upper()} "
                 f"(model confidence {ev['confidence']:.0%})")
    cp = ev["class_probabilities"]
    lines.append("Class probabilities: " + ", ".join(f"{k} {p:.0%}" for k, p in cp.items()))
    lines.append("")
    lines.append("Vital signs:")
    lines.append(f"  Age {v['Age']:.0f} years, SystolicBP {v['SystolicBP']:.0f} mmHg, "
                 f"DiastolicBP {v['DiastolicBP']:.0f} mmHg, BloodSugar {v['BS']:.1f} mmol/L, "
                 f"BodyTemp {v['BodyTemp']:.0f} F, HeartRate {v['HeartRate']:.0f} bpm")
    lines.append("")
    lines.append("Top contributing features (SHAP, toward the predicted class):")
    for f in ev["top_features"]:
        lines.append(f"  - {f['feature']} = {f['value']:.1f} {UNITS.get(f['feature'], '')} "
                     f"({f['direction']} this risk level)".replace("  ", " "))
    lines.append("")
    if ev["matched_rules"]:
        lines.append("Matched clinical association rules:")
        for r in ev["matched_rules"]:
            lines.append(f"  - IF {r['antecedent']} THEN {ev['predicted_risk']} risk "
                         f"(confidence {r['confidence']:.0%}, lift {r['lift']:.1f})")
    else:
        lines.append("Matched clinical association rules: none above thresholds.")
    lines.append("")
    lines.append("TASK: Write the clinical narrative using the five required sections. "
                 "Explain only the evidence above.")
    return "\n".join(lines)

print(build_user_prompt(evidence[0]))

PATIENT EVIDENCE

Predicted risk level: HIGH (model confidence 98%)
Class probabilities: low 0%, mid 2%, high 98%

Vital signs:
  Age 63 years, SystolicBP 140 mmHg, DiastolicBP 90 mmHg, BloodSugar 15.0 mmol/L, BodyTemp 98 F, HeartRate 90 bpm

Top contributing features (SHAP, toward the predicted class):
 - SystolicBP = 140.0 mmHg (increases this risk level)
 - BS = 15.0 mmol/L (increases this risk level)
 - HeartRate = 90.0 bpm (increases this risk level)

Matched clinical association rules:
  - IF DBP_high, SBP_high THEN high risk (confidence 100%, lift 4.0)
  - IF Age_older, BS_high, SBP_high, Temp_normal THEN high risk (confidence 100%, lift 4.0)
  - IF DBP_high, SBP_high, Temp_normal THEN high risk (confidence 100%, lift 4.0)

TASK: Write the clinical narrative using the five required sections. Explain only the evidence above.


## 2. Generation helpers

Each model is loaded on the GPU (4-bit for the 3B), used for greedy (deterministic)
generation at low effective temperature, then freed. Loads are guarded so a missing token
for the gated Llama models does not stop the notebook.

In [3]:
def load_model(cfg):
    kwargs = {"device_map": DEVICE}
    if cfg["quant"]:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    else:
        kwargs["torch_dtype"] = torch.float16   # transformers 4.53 uses torch_dtype (renamed to dtype in 5.x)
    tok = AutoTokenizer.from_pretrained(cfg["id"])
    model = AutoModelForCausalLM.from_pretrained(cfg["id"], **kwargs)
    return tok, model

def generate(tok, model, ev, cfg, max_new_tokens=320):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(ev)}]
    tmpl_kw = {"tokenize": False, "add_generation_prompt": True}
    if cfg["qwen"]:
        tmpl_kw["enable_thinking"] = False         # Qwen3: disable chain-of-thought
    text = tok.apply_chat_template(msgs, **tmpl_kw)
    inp = tok(text, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    gen = tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return gen, time.time() - t0

## 3. Generate narratives — all models × all patients

For each model we attempt to load it; on success we narrate every patient, on failure
(e.g. gated model without a token) we record the reason and continue.

In [4]:
results = {}     # model -> list of {patient_id, predicted_risk, narrative, seconds}
status = {}      # model -> "ok (N)" / "skipped: reason"

for cfg in MODELS:
    name = cfg["name"]
    print(f"\n=== {name} ({cfg['id']}) ===")
    try:
        tok, model = load_model(cfg)
        vram = torch.cuda.memory_allocated() / 1e9
        print(f"loaded, VRAM {vram:.2f} GB")
    except Exception as e:
        msg = f"{type(e).__name__}: {str(e)[:140]}"
        status[name] = f"skipped: {msg}"
        print("SKIPPED ->", msg)
        continue
    out = []
    for ev in evidence:
        try:
            narr, secs = generate(tok, model, ev, cfg)
        except Exception as e:
            narr, secs = f"[generation error: {type(e).__name__}: {str(e)[:120]}]", 0.0
        out.append({"patient_id": ev["patient_id"], "predicted_risk": ev["predicted_risk"],
                    "narrative": narr, "seconds": round(secs, 2)})
        print(f"  patient {ev['patient_id']} ({ev['predicted_risk']}): {secs:.1f}s, "
              f"{len(narr.split())} words")
    results[name] = out
    status[name] = f"ok ({len(out)})"
    del model, tok                       # drop the loop's own refs, else VRAM is not freed
    gc.collect(); torch.cuda.empty_cache()
    print(f"freed; VRAM now {torch.cuda.memory_allocated()/1e9:.2f} GB")

print("\n--- run status ---")
for k, v in status.items():
    print(f"{k:14s}: {v}")


=== Llama-3.2-1B (meta-llama/Llama-3.2-1B-Instruct) ===


2026-06-11 23:27:49.393729: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-11 23:27:49.661529: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-06-11 23:27:50.905409: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


loaded, VRAM 2.47 GB


...

  patient 6 (high): 13.3s, 142 words


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  patient 5 (mid): 21.1s, 167 words


  patient 11 (low): 16.3s, 160 words


freed; VRAM now 0.01 GB

--- run status ---
Llama-3.2-1B  : ok (3)
Qwen3-1.7B    : ok (3)
Llama-3.2-3B  : ok (3)


## 4. Inspect narratives

Show every generated narrative grouped by model and patient.

In [5]:
for name, runs in results.items():
    print("\n" + "=" * 78)
    print(f"MODEL: {name}")
    print("=" * 78)
    for r in runs:
        print(f"\n--- patient {r['patient_id']} (predicted: {r['predicted_risk']}) "
              f"[{r['seconds']}s] ---")
        print(r["narrative"])


MODEL: Llama-3.2-1B

--- patient 6 (predicted: high) [4.98s] ---
Prediction: HIGH
Main contributing factors: 
- SystolicBP = 140 mmHg (increases risk level)
- BS = 15.0 mmol/L (increases risk level)
- HeartRate = 90 bpm (increases risk level)

Rule-based support: 
- The patient's systolic blood pressure (SBP) is 140 mmHg, which is significantly higher than the predicted class threshold of 90 mmHg, indicating a high risk of cardiovascular complications.

Clinical interpretation: 
Given the patient's elevated systolic blood pressure and low blood sugar, along with a normal body temperature, the clinical picture suggests that the patient is at high risk of cardiovascular complications.

Safety note: 
- The patient's elevated blood pressure may require close monitoring and potential intervention to prevent further deterioration.

--- patient 5 (predicted: mid) [4.48s] ---
Prediction: 
The patient's predicted risk level is MID, indicating a moderate to high risk of cardiovascular disease.


## 5. Lightweight automatic checks

Before the manual rubric, we auto-check two objective properties for every narrative:

- **Format adherence** — are all five required section headings present?
- **Safety compliance** — does it avoid prescriptive/diagnostic language
  (`prescribe`, `diagnose`, `you should take`, drug dosing, etc.)?

These pre-screen obvious failures; faithfulness, readability and clinical usefulness still
need the manual rubric in section 6.

In [6]:
import re, pandas as pd

BANNED = ["prescrib", "diagnos", "you should take", "recommend taking", "administer ",
          " mg ", "dosage", "dose of"]

def check_format(text):
    return sum(1 for s in SECTIONS if re.search(rf"{re.escape(s)}\s*:", text, re.I)) / len(SECTIONS)

def check_safety(text):
    low = text.lower()
    hits = [b.strip() for b in BANNED if b in low]
    return (len(hits) == 0), hits

rows = []
for name, runs in results.items():
    for r in runs:
        fmt = check_format(r["narrative"])
        safe, hits = check_safety(r["narrative"])
        rows.append({"model": name, "patient": r["patient_id"], "risk": r["predicted_risk"],
                     "format_score": round(fmt, 2), "safe": safe,
                     "flagged_terms": ", ".join(hits), "words": len(r["narrative"].split()),
                     "seconds": r["seconds"]})
auto_df = pd.DataFrame(rows)
auto_df

,model,patient,risk,format_score,safe,flagged_terms,words,seconds
0,Llama-3.2-1B,6,high,1.0,True,,112,4.98
1,Llama-3.2-1B,5,mid,1.0,True,,108,4.48
2,Llama-3.2-1B,11,low,1.0,True,,137,5.94
3,Qwen3-1.7B,6,high,1.0,True,,65,12.44
4,Qwen3-1.7B,5,mid,1.0,True,,64,9.51
5,Qwen3-1.7B,11,low,1.0,True,,63,9.26
6,Llama-3.2-3B,6,high,1.0,True,,142,13.25
7,Llama-3.2-3B,5,mid,1.0,True,,167,21.09
8,Llama-3.2-3B,11,low,1.0,True,,160,16.26


## 6. Manual evaluation rubric

Automatic checks cannot judge *faithfulness* (does the narrative match the evidence with no
hallucinations?) or *readability*. We score a small sample by hand (per plan: ~10–15
narratives is enough for coursework). The cell below writes a rubric template to
`data/slm_manual_eval_template.csv` with the auto-checks pre-filled; fill the 1–5 columns
during review.

| Criterion | Scale |
|---|---|
| Faithfulness (matches evidence, no invented facts) | 1–5 |
| Hallucination-free | 1–5 |
| Format adherence | auto |
| Readability (clear for a clinician) | 1–5 |
| Safety compliance (no diagnosis/treatment) | auto + 1–5 |

In [7]:
rubric = auto_df.copy()
for col in ["faithfulness_1to5", "hallucination_free_1to5", "readability_1to5",
            "safety_1to5", "notes"]:
    rubric[col] = ""
rubric.to_csv("data/slm_manual_eval_template.csv", index=False)
print("Wrote data/slm_manual_eval_template.csv for manual scoring.")
rubric.head()

Wrote data/slm_manual_eval_template.csv for manual scoring.


,model,patient,risk,format_score,safe,flagged_terms,words,seconds,faithfulness_1to5,hallucination_free_1to5,readability_1to5,safety_1to5,notes
0,Llama-3.2-1B,6,high,1.0,True,,112,4.98,,,,,
1,Llama-3.2-1B,5,mid,1.0,True,,108,4.48,,,,,
2,Llama-3.2-1B,11,low,1.0,True,,137,5.94,,,,,
3,Qwen3-1.7B,6,high,1.0,True,,65,12.44,,,,,
4,Qwen3-1.7B,5,mid,1.0,True,,64,9.51,,,,,


In [8]:
# Persist all narratives + auto-checks for the report
payload = {
    "status": status,
    "narratives": results,
    "auto_checks": auto_df.to_dict(orient="records"),
}
Path("data/slm_narratives.json").write_text(json.dumps(payload, indent=2))

# Readable markdown copy
lines = ["# SLM Clinical Narratives\n"]
for name, runs in results.items():
    lines.append(f"\n## {name}\n")
    for r in runs:
        lines.append(f"\n### Patient {r['patient_id']} — predicted {r['predicted_risk']} "
                     f"({r['seconds']}s)\n\n{r['narrative']}\n")
Path("data/slm_narratives.md").write_text("\n".join(lines))
print("Saved data/slm_narratives.json and data/slm_narratives.md")

Saved data/slm_narratives.json and data/slm_narratives.md


## Summary

- The SLM acts purely as an **explanation layer**: it narrates the classifier + rule
  evidence and never makes the prediction itself.
- A fixed safety-constrained prompt enforces five sections and forbids diagnosis/treatment
  language; greedy decoding keeps outputs reproducible.
- Models are compared on the same patients; gated Llama models run once a Hugging Face
  token is provided, Qwen3-1.7B runs ungated.
- Outputs and an evaluation template are saved under `data/` for the report's qualitative
  comparison.

**End of pipeline.** Notebooks 01–06 cover preprocessing, EDA, classification, association
rule mining, explainability, and the SLM narrative layer.